In [1]:
# 1. Download the repository from GitHub
!git clone https://github.com/Ibraheem-Al-hafith/miniGPT.git

# 2. Change the directory to the newly downloaded folder
# (Notice we use % instead of ! for 'cd' in Colab so the folder change sticks)
%cd miniGPT


Cloning into 'miniGPT'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 118 (delta 52), reused 96 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 2.26 MiB | 28.59 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/miniGPT


In [23]:
from data.dataset import download_and_load_file , data_split, get_instruction_loaders , format_input
from inference.load_weights import load_from_hf
from train.trainer import calc_loss_loader , train, _save_loss_plot
from inference.generate import generate, _encode, _decode
from config import MAX_LEN
import torch

In [4]:
data = download_and_load_file("instruction_data.json","https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/instruction-data.json")
model,config= load_from_hf("gpt2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded gpt2 — 163,037,184 parameters


In [5]:
!ls

assets	     data		    __pycache__        tests
checkpoints  inference		    pyproject.toml     train
cli.py	     instruction_data.json  README.md	       ui
config.py    model		    sair_gpt_demo.gif  uv.lock


In [6]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
train ,test,val = data_split("instruction_data.json")


Splits saved: train.json, test.json, val.json in '.'


In [7]:
!ls 


assets	     data		    __pycache__        test.json   ui
checkpoints  inference		    pyproject.toml     tests	   uv.lock
cli.py	     instruction_data.json  README.md	       train	   val.json
config.py    model		    sair_gpt_demo.gif  train.json


In [8]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
train_loader , val_loader,test_loader  = get_instruction_loaders(
    "",
    tokenizer
)


(tensor([[21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 30003,  6525,   262,  1708,  6827,   284,
          2291,   257,  8718, 45693,    25,   705,    40,  1101,   845, 10032,
          2637,   198,   198, 21017, 23412,    25,   220,   198,    40,  1101,
           845, 10032,    13,   198,   198, 21017, 18261,    25,   220,   198,
            40,  1101,   523, 10032,   314,   714,  3993,   329,   257,   614,
            13],
        [21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,  5376,   262,  1115, 13737,   286,   262,
          1578,  1829,  1230,    13,   198,   198, 21017, 18261,    25,   220,
           198,   464,  1115, 13737,   286,   262,  1578,  1829,  1230,   389,
           262, 10390,    11,   26

In [20]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 4.233151817321778
Validation loss: 4.127959585189819


In [ ]:
train_losses, val_losses, tokens_seen = train(
    model, train_loader, val_loader, device ,start_context=format_input(val[0]), tokenizer=tokenizer   
)

In [ ]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)


In [ ]:
#For evaluation
torch.manual_seed(123)

for entry in test[:3]:
    input_text = format_input(entry)
    generated_text = generate(
        model = model,
        prompt = input_text,
        max_new_tokens= 256,
        context_size=MAX_LEN,
        tokenizer =tokenizer,
        device=device
    )
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")